# A gentle introduction to `torch.autograd`

`torch.autograd` is PyTorch's automatic diffeerentiation engine that powers neural network training.
In this section, you will get a concepptual understanding of how autograd helps a neural network train.


## Background

Neural networks (NNs) are a collection of *nested funcitons* that are executed on some input data. These functions are defined by parameters (consisting of weights and biases), which in PyTorch are stored in tensors.

Training a NN happens in two steps:

**Forward Propagation**: In forward prop, the NN makes its best guess about the correct output. It runs the input data through each of its functions to make this guess.

**Backward Propagation**: In backprop, the NN adjusts its parameters proportionate to the error in its guess. It does this by traversing backwards from the output, collecting the derivatives of the error with respect to the parameters of the functions (gradients), and optimizing the parameters using gradient descent. For a more detailed walkthrough of backprop, check out this [video from 3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).
> traverse: to go through, across, or over an area.

## Usage in Pytorch

Let's look at a single training step. For this example, we load a pre-trained ResNet-18 model from `torchvision`. We create a random data tensor to represent a single image with three channels and height and width of 64, and its corresponding label initialized to some random values. The label in the pre-trained model has shape (1, 1000).

In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/kasm-user/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:04<00:00, 11.5MB/s]


Next, we run the input data through the model, through each of its layers, to make a prediction. This is the forward pass.

In [2]:
prediction = model(data) # forward pass

We use the model's prediction and the corresponding label to calculate the error loss. The next step is to backpropagate this error through the network. 

Backward propagation is kicked off when we call `.backward()` on the error tensor. Autograd then calculates and stores the gradients for each model parameter in the parameter's `.grad` attribute.

In [3]:
loss = (prediction - labels).sum()
loss.backward() # backward pass

Next, we load an optimizer, in this case SGD with a learning rate of 0.01 and momentum of 0.9. We register all the parameters of the model in the optimizer.

In [4]:
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

Finally, we call `.step()` to initiate gradient descent. The optimizer adjusts each parameter by its gradient stored in `.grad`.

In [5]:
optim.step() #gradient descent

## Differentiation in Autograd

Let's take a look at how `autograd` collects gradients. We create two tensors `a` and `b` with `requires_grad=True`. This signals to `autograd` that every operation on them should be tracked.

In [9]:
import torch

a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

We create another tensor `Q` from `a` and `b`.
$$Q=3a^3-b^2$$

In [10]:
Q=3*a**3-b**2
Q

tensor([-12.,  65.], grad_fn=<SubBackward0>)

Let's assume `a` and `b` to be parameters of an NN, and `Q` to be the error. In NN training, we want gradients of the error w.r.t. parameters, i.e.
$$
\frac{\partial{Q}}{\partial{a}}=9a^2
$$
$$
\frac{\partial{Q}}{\partial{b}}=-2b
$$

When we call `.backward()` on `Q`, autograd calculates these gradients and stores them in the respective tensors' `.grad` attribute.

We need to explicitly pass a `gradient` argument in `Q.backward()` because it is a vecctor. `gradient` is a tensor of the same shape as `Q`, and it represents the gradient of `Q` w.r.t. itself, i.e.
$$
\frac{\partial{Q}}{\partial{Q}}=1
$$

Equivalently, we can also aggregate `Q` into a scalar and call backward implicitly, like `Q.sum().backward()`.

In [11]:
external_grad = torch.tensor([1., 1.])
Q.backward(gradient=external_grad)

Gradients are now deposited in `a.grad` and `b.grad`

In [12]:
# check if collected gradients are correct
print(9*a**2==a.grad)
print(-2*b==b.grad)

tensor([True, True])
tensor([True, True])


## Vector Calculate using `autograd`

Mathematically, if you have a vector-valued function $\vec{y}=f(\vec{x})$, then the gradient of $\vec{y}$ with respect to $\vec{x}$ is a Jacobian matrix $J$:

> valued: having a particular type of value as its output. A vector-valued function is a function whose output is a vector rather than a single number.